# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:


# Write your code below.
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(
    os.path.join(PRICE_DATA, "**", "*.parquet"),
    recursive=True
)

print("PRICE_DATA:", PRICE_DATA)
print("Number of Parquet files:", len(parquet_files))


PRICE_DATA: ../../05_src/data/prices/
Number of Parquet files: 2045


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:

dd_prices = dd.read_parquet(parquet_files)
print(dd_prices.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year']


In [19]:
# Write your code below.
import pandas as pd

# Read parquet data with Dask
dd_prices = dd.read_parquet(parquet_files)

# Define the feature engineering operation for each stock
def create_features(df):
    df = df.sort_values('Date').copy()
    df['Close_lag_1'] = df['Close'].shift(1)
    df['Adj_Close_lag_1'] = df['Adj Close'].shift(1)
    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
    df['hi_lo_range'] = df['High'] - df['Low']
    return df

# Group by ticker and create features within each stock
# Preserve ticker when creating features for each stock
def create_features_with_ticker(df):
    ticker = df.name
    result = create_features(df)
    result['ticker'] = ticker
    return result

meta = create_features(dd_prices._meta.drop(columns=['ticker']))
meta['ticker'] = pd.Series(dtype=dd_prices._meta['ticker'].dtype)

dd_feat = dd_prices.groupby(
    'ticker', group_keys=False
).apply(
    create_features_with_ticker,
    meta=meta
)

In [20]:
print("Ticker column exists:", 'ticker' in dd_feat.columns)
print("Total rows:", dd_feat.map_partitions(len).sum().compute())

Ticker column exists: True
Total rows: 239659


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [21]:

# Convert Dask DataFrame to Pandas
df_pandas = dd_feat.compute()

# Sort by ticker and date
df_pandas = df_pandas.sort_values(['ticker', 'Date'])

# Calculate the 10-day moving average for each ticker
df_pandas['returns_ma_10'] = (
    df_pandas.groupby('ticker')['returns']
    .transform(lambda x: x.rolling(window=10).mean())
)

# Display the results
print(df_pandas[['Date', 'ticker', 'returns', 'returns_ma_10']].tail(15))


            Date ticker   returns  returns_ma_10
88133 2020-03-12   ZIXI -0.185309      -0.045392
88134 2020-03-13   ZIXI -0.030738      -0.046965
88135 2020-03-16   ZIXI -0.103594      -0.060751
88136 2020-03-17   ZIXI -0.075472      -0.065108
88137 2020-03-18   ZIXI -0.227041      -0.092628
88138 2020-03-19   ZIXI  0.095710      -0.080155
88139 2020-03-20   ZIXI  0.069277      -0.066503
88140 2020-03-23   ZIXI -0.090141      -0.065370
88141 2020-03-24   ZIXI  0.142415      -0.053803
88142 2020-03-25   ZIXI  0.084011      -0.032088
88143 2020-03-26   ZIXI  0.127500      -0.000807
88144 2020-03-27   ZIXI  0.019956       0.004262
88145 2020-03-30   ZIXI  0.008696       0.015491
88146 2020-03-31   ZIXI -0.071121       0.015926
88147 2020-04-01   ZIXI -0.113689       0.027261


In [12]:
# Write your code below.

# Create features for each ticker
def create_features(df):
    df = df.sort_values('Date').copy()

    df['Close_lag_1'] = df['Close'].shift(1)
    df['Adj_Close_lag_1'] = df['Adj Close'].shift(1)

    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
    df['hi_lo_range'] = df['High'] - df['Low']

    return df

# Group by ticker and preserve ticker as a regular column
dd_feat = dd_prices.groupby(
    'ticker', group_keys=False
).apply(
    create_features,
    meta=create_features(dd_prices._meta.drop(columns=['ticker']))
)

In [13]:
print("Total rows:", dd_feat.map_partitions(len).sum().compute())

Total rows: 239659


In [15]:
print("Total rows:", dd_feat.map_partitions(len).sum().compute())
print("Number of partitions:", dd_feat.npartitions)

print("First 10 rows:")
print(dd_feat.head(10, npartitions=-1))

Total rows: 239659
Number of partitions: 2045
First 10 rows:
             Date       Open       High        Low      Close  Adj Close  \
217418 2016-01-08  19.162428  19.162428  19.114038  19.114038  11.411459   
217419 2016-01-11  19.133394  19.230173  19.017258  19.017258  11.353675   
217420 2016-01-12  19.113070  19.113070  19.103392  19.103392  11.405104   
217421 2016-01-13  19.103392  19.103392  19.103392  19.103392  11.405104   
217422 2016-01-14  19.065647  19.385021  18.968868  19.104361  11.405683   
217423 2016-01-15  19.181784  19.181784  18.937899  18.937899  11.306301   
217424 2016-01-19  19.267918  20.459278  18.894348  18.894348  11.280302   
217425 2016-01-20  18.872087  18.876926  18.630138  18.630138  11.122562   
217426 2016-01-21  18.630138  18.630138  18.630138  18.630138  11.122562   
217427 2016-01-22  18.930157  18.930157  18.792728  18.792728  11.219632   

         Volume   source  Year  Close_lag_1  Adj_Close_lag_1   returns  \
217418  10300.0  RIV.csv  20

In [16]:
print("Columns:", dd_feat.columns.tolist())
print("Index names:", dd_feat._meta.index.names)

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range']
Index names: [None]


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not strictly necessary to convert the Dask DataFrame to Pandas to calculate the 10-day moving average. However, Pandas made the calculation simpler because it supports `groupby()` and `rolling()` operations directly.

Using Dask could be more efficient for much larger datasets because it supports parallel processing and does not require loading the entire dataset into memory. However, calculating a rolling average in Dask requires careful handling of partition boundaries to ensure accurate results. For this assignment, Pandas was a practical choice because the dataset could fit into memory.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.